In [1]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
import random
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

from tqdm.notebook import tqdm
import seaborn as sns
from collections import Counter

from glob import glob
import psi4
from helper_CC_ML_spacial import *

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

  Threads set to 12 by Python driver.


In [2]:
# Most important features (top 5 by SHAP):
# doublecheck: Numerator of the MP2 t2-amplitude, two-electron integral <ik || ab>
# t2start: Initial MP2 t2-amplitude
# t2mag: Magnitude of the MP2 t2-amplitude
# orbdiff: Denominator of the MP2 t2-amplitude
# diag: Binary feature denoting whether a=b (virtual orbits are the same)
top5 = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']

# 31 Features order in X, including t2
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

In [3]:
basis_sets = ['STO-3G', 'cc-pVDZ', 'aug-cc-pVDZ']

In [4]:
molecules = ["water", "methanol", "ethylene", "ethane", "methane", "ammonia", "formaldehyde"]

In [5]:
random.seed(0)
all_sampled_files = []
for mol in molecules:
    all_files = sorted(glob(os.path.join("data", f"{mol}*.xyz")))
    sample_size = min(100, len(all_files))
    sampled_files = random.sample(all_files, sample_size)

    all_sampled_files.extend(sampled_files)  # append sampled files to a list

    print(f"Total number of {mol} files: {len(all_files)}")
    print(f"{mol}: {len(sampled_files)} files sampled")
    for f in sampled_files:
        print(f)
    print("\n")

Total number of water files: 201
water: 100 files sampled
data/water189.xyz
data/water93.xyz
data/water197.xyz
data/water109.xyz
data/water16.xyz
data/water35.xyz
data/water3.xyz
data/water193.xyz
data/water17.xyz
data/water28.xyz
data/water182.xyz
data/water52.xyz
data/water15.xyz
data/water34.xyz
data/water131.xyz
data/water165.xyz
data/water86.xyz
data/water121.xyz
data/water60.xyz
data/water158.xyz
data/water40.xyz
data/water57.xyz
data/water133.xyz
data/water171.xyz
data/water122.xyz
data/water116.xyz
data/water176.xyz
data/water26.xyz
data/water47.xyz
data/water77.xyz
data/water181.xyz
data/water20.xyz
data/water172.xyz
data/water59.xyz
data/water65.xyz
data/water147.xyz
data/water45.xyz
data/water90.xyz
data/water201.xyz
data/water38.xyz
data/water95.xyz
data/water113.xyz
data/water44.xyz
data/water102.xyz
data/water120.xyz
data/water192.xyz
data/water10.xyz
data/water31.xyz
data/water177.xyz
data/water156.xyz
data/water175.xyz
data/water114.xyz
data/water143.xyz
data/water49.xy

In [6]:
all_sampled_files[:5]

['data/water189.xyz',
 'data/water93.xyz',
 'data/water197.xyz',
 'data/water109.xyz',
 'data/water16.xyz']

In [7]:
random.seed(0)
train_size = min(100, len(all_sampled_files))
train = random.sample(all_sampled_files, train_size)

print(f"Total sampled files: {len(all_sampled_files)}")
print(f"Train set size: {len(train)}")

Total sampled files: 698
Train set size: 100


In [8]:
random.seed(0)
remaining_files = list(set(all_sampled_files) - set(train))
test_size = min(50, len(remaining_files))
test = random.sample(remaining_files, test_size)

print(f"Total sampled files: {len(all_sampled_files)}")
print(f"Train set size: {len(train)}")
print(f"Test set size: {len(test)}")

Total sampled files: 698
Train set size: 100
Test set size: 50


In [9]:
train[:5], test[:5]

(['data/ethane179.xyz',
  'data/methane95.xyz',
  'data/water113.xyz',
  'data/ethylene66.xyz',
  'data/ammonia72.xyz'],
 ['data/ethane140.xyz',
  'data/ethylene53.xyz',
  'data/methane173.xyz',
  'data/formaldehyde18.xyz',
  'data/methane104.xyz'])

In [16]:
train[-5:], test[-5:]

(['data/methanol14.xyz',
  'data/water38.xyz',
  'data/formaldehyde67.xyz',
  'data/formaldehyde168.xyz',
  'data/methane68.xyz'],
 ['data/ammonia98.xyz',
  'data/methane85.xyz',
  'data/methane26.xyz',
  'data/methane92.xyz',
  'data/methanol72.xyz'])

In [17]:
with open("out/train_names.txt", "w") as f:
    for molecule in train:
        f.write(molecule + "\n")

In [18]:
with open("out/test_names.txt", "w") as f:
    for molecule in test:
        f.write(molecule + "\n")

In [ ]:
################################################################

In [25]:
with open("out/train_names.txt", "r") as f:
    lines = f.readlines()

train = [element[:-1] for element in lines]

In [26]:
with open("out/test_names.txt", "r") as f:
    lines = f.readlines()

test = [element[:-1] for element in lines]

In [27]:
train[:5], test[:5], train[-5:], test[-5:]

(['data/ethane179.xyz',
  'data/methane95.xyz',
  'data/water113.xyz',
  'data/ethylene66.xyz',
  'data/ammonia72.xyz'],
 ['data/ethane140.xyz',
  'data/ethylene53.xyz',
  'data/methane173.xyz',
  'data/formaldehyde18.xyz',
  'data/methane104.xyz'],
 ['data/methanol14.xyz',
  'data/water38.xyz',
  'data/formaldehyde67.xyz',
  'data/formaldehyde168.xyz',
  'data/methane68.xyz'],
 ['data/ammonia98.xyz',
  'data/methane85.xyz',
  'data/methane26.xyz',
  'data/methane92.xyz',
  'data/methanol72.xyz'])

## Note: we do X_test first because we do not adjust the size of this, we test each trained model on the same test set

# X_test

In [28]:
X_test_all = {}
y_test_all = {}

for basis in basis_sets[:2]: 
    filenames = test.copy()
    t1 = time.time()
    
    print(basis)

    data_dict = {}
    for fn in filenames:
        struct = os.path.basename(fn)
        print(f"Processing {struct}")
        
        with open(fn,'r') as f:
            text=f.read()
        
        mol = psi4.geometry(text)
        
        psi4.core.clean()
        psi4.core.be_quiet()
        
        psi4.set_options({'basis': basis,
                          'scf_type': 'pk',
                          'reference': 'rohf',
                          'mp2_type': 'conv',
                          'e_convergence': 1e-8,
                          'd_convergence': 1e-8})

        try:
            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            
            A = HelperCCEnergy(mol, rhf_e, scf_wfn, freeze_core=False)

            MP2T2=A.t2start                                                            # CAN YOU LET ME KNOW IF THIS IS CORRECT
            A.t1 = np.zeros((A.t1.shape))
            A.t2 = MP2T2
            
            MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
            CCSDE = A.compute_energy()                                   # exact CCSD energy

            data = pd.DataFrame(np.array([getattr(A, attr).flatten() for attr in properties]).T, columns=properties)
            data_dict[struct.split('_')[0]] = data

        except Exception as e:
            print(f"Molecule with filename {fn} failed: {e}")
            pass   

    X_test_all[basis] = np.vstack([df[top5].to_numpy() for df in data_dict.values()])
    y_test_all[basis] = np.concatenate([df["t2"].to_numpy().reshape(-1) for df in data_dict.values()])
    t2 = time.time()

    print(f"Time taken for {basis}: {t2-t1} seconds \n")

STO-3G
Processing ethane140.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.282 seconds.

CCSD Iteration   0: CCSD correlation = -0.123998511075192   dE =  1.23999E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.123998511075192   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.006 seconds!
CCSD Iteration   0: CCSD correlation = -0.123998511075192   dE =  1.23999E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.147143755434082   dE = -2.31452E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.155350641674608   dE = -8.20689E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.161359482219436   dE = -6.00884E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.162353009613518   dE = -9.93527E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.162442352692769   dE = -8.93431E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.16242263699842

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.117 seconds.

CCSD Iteration   0: CCSD correlation = -0.123452970094771   dE =  1.23453E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.123452970094771   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.123452970094771   dE =  1.23453E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.146610821990746   dE = -2.31579E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.154795150666891   dE = -8.18433E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.160749645262298   dE = -5.95449E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.161726837361294   dE = -9.77192E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.161816061089926   dE = -8.92237E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.16179681502808

...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 34 basis functions.
(34, 34)
(34, 34)
Building initial guess...

..initialized CCSD in 0.531 seconds.

CCSD Iteration   0: CCSD correlation = -0.164265751051522   dE =  1.64266E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.164265751051522   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.164265751051522   dE =  1.64266E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.181764921155881   dE = -1.74992E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.185907809827896   dE = -4.14289E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.187401186339667   dE = -1.49338E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.187613162910610   dE = -2.11977E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.187636511456656   dE = -2.33485E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.18763729085168

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 34 basis functions.
(34, 34)
(34, 34)
Building initial guess...

..initialized CCSD in 0.499 seconds.

CCSD Iteration   0: CCSD correlation = -0.164170796847256   dE =  1.64171E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.164170796847256   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.164170796847256   dE =  1.64171E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.181659326812675   dE = -1.74885E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.185787374775502   dE = -4.12805E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.187272045955816   dE = -1.48467E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.187481399288578   dE = -2.09353E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.187504394848237   dE = -2.29956E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.18750514657435

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 48 basis functions.
(48, 48)
(48, 48)
Building initial guess...

..initialized CCSD in 2.373 seconds.

CCSD Iteration   0: CCSD correlation = -0.342480863875085   dE =  3.42481E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.342480863875085   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = -0.342480863875085   dE =  3.42481E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.357214629670454   dE = -1.47338E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.364591509464356   dE = -7.37688E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.366282814811757   dE = -1.69131E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.366961715295556   dE = -6.78900E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.367023830316854   dE = -6.21150E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.36703254857298

# X_train

## SEE THE CELL BELOW THIS CELL FOR THE HYPERPARAMETER FINETUNED VERSION

In [12]:
sizes = [10, 20, 40, 60, 80, 100]
X_train_all = {}
y_train_all = {}

for basis in basis_sets: 
    for n in sizes:
        filenames = train[:n]
        t1 = time.time()
        
        print(f"{basis} Basis, N = {n} training molecules")
        print(f"Training molecules: {filenames}")

        # get training molecule
        data_dict = {}
        for fn in filenames:
            struct = os.path.basename(fn)
            print(f"Processing {struct}")
            
            with open(fn,'r') as f:
                text=f.read()
            
            mol = psi4.geometry(text)
            
            psi4.core.clean()
            psi4.core.be_quiet()
            
            psi4.set_options({'basis': basis,
                              'scf_type':     'pk',
                              'reference':    'rohf',
                              'mp2_type':     'conv',
                              'e_convergence': 1e-8,
                              'd_convergence': 1e-8})

            try:
                
                rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                
                A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
    
    
                MP2T2=A.t2start
                A.t1 = np.zeros((A.t1.shape))
                A.t2 = MP2T2
                
                MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
                CCSDE = A.compute_energy()                                   # exact CCSD energy
    
                data=pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
                data_dict[struct.split('_')[0]]=data
            except Exception as e:
                print(f"Molecule with filename {fn} failed: {e}")
                pass   

        X_train_all[basis] = np.vstack([df[top5].to_numpy() for df in data_dict.values()])
        y_train_all[basis] = np.concatenate([df["t2"].to_numpy().reshape(-1) for df in data_dict.values()])
        
        model = XGBRegressor(
            n_estimators=400,
            max_depth=12,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            reg_alpha=0.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=42
        )

        scaler = MinMaxScaler(feature_range=(-1,1))

        X_train_scaled = scaler.fit_transform(X_train_all[basis])
        X_test_scaled = scaler.transform(X_test_all[basis])

        model.fit(X_train_scaled, y_train_all[basis])
        y_pred = model.predict(X_test_scaled)
        
        r2 = r2_score(y_test_all[basis], y_pred)
        mae = mean_absolute_error(y_test_all[basis], y_pred)
        rmse = root_mean_squared_error(y_test_all[basis], y_pred)
    
        print(f"N: {n}, MAE: {mae}, RMSE: {rmse}, R2: {r2}")

        with open("out/performance.txt", "a") as f:
            f.write(f"Basis: {basis}, N: {n}, MAE: {mae}, RMSE: {rmse}, R2: {r2}\n")
        
        joblib.dump(scaler, f"out/{basis}_scaler_{n}.pkl")
        model.save_model(f"out/{basis}_model_{n}.json")

        print("Saved file, saved scaler.")
        
        print("\n")
    print("\n")

STO-3G Basis, N = 10 training molecules
Training molecules: ['data/ethane179.xyz', 'data/methane95.xyz', 'data/water113.xyz', 'data/ethylene66.xyz', 'data/ammonia72.xyz', 'data/methane175.xyz', 'data/methane23.xyz', 'data/ethane70.xyz', 'data/methane7.xyz', 'data/ethane116.xyz']
Processing ethane179.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.103 seconds.

CCSD Iteration   0: CCSD correlation = -0.122817126293522   dE =  1.22817E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.122817126293522   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.122817126293522   dE =  1.22817E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.145878015707461   dE = -2.30609E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.154004387607901   dE = -8.12637E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.159905214690172   dE = -5.90083E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.160874133318693   dE = -9.68919E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.160960200359837   dE = -8.60670E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.16094086703658

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(9, 9)
Building initial guess...

..initialized CCSD in 0.035 seconds.

CCSD Iteration   0: CCSD correlation = -0.056770293259456   dE =  5.67703E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.056770293259456   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.056770293259456   dE =  5.67703E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.071664845091554   dE = -1.48946E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.076484637096333   dE = -4.81979E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.079189329379724   dE = -2.70469E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.079389641209204   dE = -2.00312E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.079407894426170   dE = -1.82532E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.079407133309259   d

...

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 34 basis functions.
(34, 34)
(34, 34)
Building initial guess...

..initialized CCSD in 0.661 seconds.

CCSD Iteration   0: CCSD correlation = -0.164137154725369   dE =  1.64137E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.164137154725369   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.164137154725369   dE =  1.64137E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.181622020705352   dE = -1.74849E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.185744926032319   dE = -4.12291E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.187226567331128   dE = -1.48164E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.187435011108554   dE = -2.08444E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.187457884820279   dE = -2.28737E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.18745862711861

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 29 basis functions.
(29, 29)
(29, 29)
Building initial guess...

..initialized CCSD in 0.573 seconds.

CCSD Iteration   0: CCSD correlation = -0.189039202156209   dE =  1.89039E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.189039202156209   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.003 seconds!
CCSD Iteration   0: CCSD correlation = -0.189039202156209   dE =  1.89039E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.200010176698307   dE = -1.09710E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.203564854523474   dE = -3.55468E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.204772715508644   dE = -1.20786E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.205007368825808   dE = -2.34653E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.205031865412726   dE = -2.44966E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.20503287980865

KeyboardInterrupt: 

# Hyperparameter tuning

In [32]:
sizes = [1, 5, 10, 20, 40, 60, 80, 100]
X_train_all = {}
y_train_all = {}

for basis in basis_sets: 
    for n in sizes:
        filenames = train[:n]
        t1 = time.time()
        
        print(f"{basis} Basis, N = {n} training molecules")
        print(f"Training molecules: {filenames}")

        # get training molecule
        data_dict = {}
        for fn in filenames:
            struct = os.path.basename(fn)
            print(f"Processing {struct}")
            
            with open(fn,'r') as f:
                text=f.read()
            
            mol = psi4.geometry(text)
            
            psi4.core.clean()
            psi4.core.be_quiet()
            
            psi4.set_options({'basis': basis,
                              'scf_type':     'pk',
                              'reference':    'rohf',
                              'mp2_type':     'conv',
                              'e_convergence': 1e-8,
                              'd_convergence': 1e-8})

            try:
                
                rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                
                A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
    
    
                MP2T2=A.t2start
                A.t1 = np.zeros((A.t1.shape))
                A.t2 = MP2T2
                
                MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
                CCSDE = A.compute_energy()                                   # exact CCSD energy
    
                data=pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
                data_dict[struct.split('_')[0]]=data
            except Exception as e:
                print(f"Molecule with filename {fn} failed: {e}")
                pass   

        X_train_all[basis] = np.vstack([df[top5].to_numpy() for df in data_dict.values()])
        y_train_all[basis] = np.concatenate([df["t2"].to_numpy().reshape(-1) for df in data_dict.values()])


        pipeline = Pipeline([
            ('scaler', MinMaxScaler(feature_range=(-1, 1))),
            ('xgb', XGBRegressor(tree_method="hist", n_jobs=-1, random_state=42))
        ])

        param_grid = {
            'xgb__n_estimators': [100, 200, 400],
            'xgb__max_depth': [12],
            'xgb__learning_rate': [0.01, 0.05, 0.1],
            #'xgb__subsample': [0.6, 0.8, 1.0],
            #'xgb__colsample_bytree': [0.6, 0.8, 1.0],
            #'xgb__reg_lambda': [1, 5, 10],
            #'xgb__reg_alpha': [0, 0.1, 1]
        }


        ###########
        # This is the model that we were rocking with before (see above cell)
        
        #    model = XGBRegressor(
        #        n_estimators=400,
        #        max_depth=12,
        #        learning_rate=0.05,
        #        subsample=0.8,
        #        colsample_bytree=0.8,
        #        reg_lambda=1.0,
        #        reg_alpha=0.0,
        #        tree_method="hist",
        #        n_jobs=-1,
        #        random_state=42
        #    )
        #
        # Couple of notes here:
        # 1. The params here seem to give a bigger model (in terms of storage)
        # 2. The params here give better performance than the best model from grid search, perhaps this is from the other params being tuned

        ##############
        # THIS IS GRIER'S CODE:
        #
        # params = {'max_depth': [1, 10, 100],
        #           'n_estimators': [100, 500, 1000],
        #           'reg_lambda': [1e-6, 1e-3,1e-1],
        #           'reg_alpha': [1e-6, 1e-3,1e-1]}
        
        # model = XGBRegressor()
        # grid = GridSearchCV(estimator=model, 
        #                    param_grid=params,
        #                    scoring='r2', 
        #                    verbose=1000,n_jobs=12).fit(X_train,y_train)
        
        
        # model=grid.best_estimator_
        # y_pred_train=model.predict(X_train_all[basis])
        # y_pred_test=model.predict(X_test_all[basis])
        # print(f"R2: {r2_score(y_train,y_pred_train):.4f},{r2_score(y_test,y_pred_test):.4f}")
        # print(f"RMSE (mEh): {root_mean_squared_error(y_train,y_pred_train)*1e3:.4f},{root_mean_squared_error(y_test,y_pred_test)*1e3:.4f}")
        
        # END OF GRIER'S CODE
        ##############
        
        grid_search = GridSearchCV(
            pipeline,
            param_grid,
            cv=5,
            scoring='neg_mean_absolute_error',
            verbose=10000,
            n_jobs=-1
        )

        t1 = time.time()
        grid_search.fit(X_train_all[basis], y_train_all[basis])
        t2 = time.time()
        
        y_pred = grid_search.predict(X_test_all[basis])

        print(f"Best parameters for basis {basis}, N={n}: {grid_search.best_params_}")
        
        r2 = r2_score(y_test_all[basis], y_pred)
        mae = mean_absolute_error(y_test_all[basis], y_pred)
        rmse = root_mean_squared_error(y_test_all[basis], y_pred)
    
        print(f"N: {n}, MAE: {mae}, RMSE: {rmse}, R2: {r2}, Training time: {t2-t1} sec")

        with open("out/optimised_performance.txt", "a") as f:
            f.write(f"Basis: {basis}, N: {n}, MAE: {mae}, RMSE: {rmse}, R2: {r2}\n")
        
        joblib.dump(grid_search.best_estimator_, f"out/optimised_{basis}_best_model_{n}.pkl")

        print("Saved file, saved scaler. \n")
        
        print("\n")
    print("\n")

STO-3G Basis, N = 1 training molecules
Training molecules: ['data/ethane179.xyz']
Processing ethane179.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 1.127 seconds.

CCSD Iteration   0: CCSD correlation = -0.122817126293522   dE =  1.22817E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.122817126293522   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.035 seconds!
CCSD Iteration   0: CCSD correlation = -0.122817126293522   dE =  1.22817E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.145878015707461   dE = -2.30609E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.154004387607901   dE = -8.12637E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.159905214690172   dE = -5.90083E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.160874133318692   dE = -9.68919E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.160960200359837   dE = -8.60670E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.16094086703658

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.607 seconds.

CCSD Iteration   0: CCSD correlation = -0.122817126293522   dE =  1.22817E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.122817126293522   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.010 seconds!
CCSD Iteration   0: CCSD correlation = -0.122817126293522   dE =  1.22817E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.145878015707461   dE = -2.30609E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.154004387607901   dE = -8.12637E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.159905214690172   dE = -5.90083E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.160874133318692   dE = -9.68919E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.160960200359837   dE = -8.60670E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.16094086703658

...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 12 basis functions.
(12, 12)
(12, 12)
Building initial guess...

..initialized CCSD in 0.146 seconds.

CCSD Iteration   0: CCSD correlation = -0.119677080838037   dE =  1.19677E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.119677080838037   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.119677080838037   dE =  1.19677E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.133848723191333   dE = -1.41716E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.140683408435953   dE = -6.83469E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.143442917473799   dE = -2.75951E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.144861665085510   dE = -1.41875E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.145105848974021   dE = -2.44184E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.14510644943295

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 12 basis functions.
(12, 12)
(12, 12)
Building initial guess...

..initialized CCSD in 0.055 seconds.

CCSD Iteration   0: CCSD correlation = -0.119834331604359   dE =  1.19834E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.119834331604359   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.119834331604359   dE =  1.19834E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.133948285086638   dE = -1.41140E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.140810050676507   dE = -6.86177E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.143543082496583   dE = -2.73303E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.144978887868099   dE = -1.43581E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.145224568492189   dE = -2.45681E-04   DIIS = 4
[CV 5/5; 4/9] START xgb__learning_rate=0.05, xgb__max_de

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x767963db9150>>
Traceback (most recent call last):
  File "/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
KeyboardInterrupt: 


KeyboardInterrupt: 